In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from dotenv import load_dotenv
load_dotenv()

/var/folders/fq/kr6gv5l17pd4j572n0_n3f2c0000gn/T/ipykernel_41952/3968643131.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/rahultiwari/Documents/02_Freelancing/Hachion_batch/ai_engineering_19th_may/ai-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
loader=PyPDFLoader("data_test/hr_policy_manual.pdf")
pages = loader.load()

splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ".", ","],
    chunk_size=512,
    chunk_overlap=100,
)

chunks = splitter.split_documents(pages)


embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(chunks, embeddings)

In [3]:
ret = vectorstore.as_retriever(search_kwargs={"k": 3})
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [4]:
ret.invoke("what is the notice period for senior managers?")

[Document(id='20a2c0c5-0f19-450b-b699-d369ef0ec984', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-31T19:53:57+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-31T19:53:57+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'data_test/hr_policy_manual.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='manager and HR Business Partner.\n1.2 Notice Period\nPost-confirmation, the standard notice period is 60 days for all employees up to Senior Manager\nlevel. For Associate Director and above, the notice period is 90 days. Notice period buyout is\npermitted at basic salary per day, subject to management approval.\n1.3 Working Hours\nStandard working hours are 9 hours per day, Monday to Friday. Core hours are 10:00 AM to 5:00\nPM IST. Flexible start is permitted between 8:00 AM and 10:30 AM. Work from Home (WFH) is'),
 Document(id='94d6cb21-ef03-40

In [6]:
prompt = ChatPromptTemplate.from_template(
       "Answer using context only.\nContext: {context}\nQuestion: {question}"
)

def fmt(docs):
    return "\n\n".join([doc.page_content for doc in docs])


chain = (
    {"context": ret | fmt, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

In [7]:
def ask(q):
    return chain.invoke(q), ret.invoke(q)

In [8]:
ans, src = ask("What is the notice period for senior managers?")

In [9]:
ans

'The notice period for senior managers is 60 days.'

In [10]:
for i, d in enumerate(src,1):
    print("###")
    print(f"{i}. {d.page_content}")

###
1. manager and HR Business Partner.
1.2 Notice Period
Post-confirmation, the standard notice period is 60 days for all employees up to Senior Manager
level. For Associate Director and above, the notice period is 90 days. Notice period buyout is
permitted at basic salary per day, subject to management approval.
1.3 Working Hours
Standard working hours are 9 hours per day, Monday to Friday. Core hours are 10:00 AM to 5:00
PM IST. Flexible start is permitted between 8:00 AM and 10:30 AM. Work from Home (WFH) is
###
2. NovaTech Solutions Pvt. Ltd.
Employee HR Policy Manual — FY 2024-25 | Version 3.2 | Confidential
Section 1: Employment Terms
1.1 Probation Period
All new employees at NovaTech Solutions are subject to a probation period of 6 months from their
date of joining. During the probation period, either party may terminate employment with a notice
period of 15 days. Confirmation is subject to a satisfactory performance review by the reporting
manager and HR Business Partner.
1.2 

In [ ]:
ans2, srcs2 = ask("please summarize the entire documents")


'The document outlines the paternity leave policy and the salary structure at NovaTech. The salary structure includes various components: Basic Salary (40% of CTC, fully taxable), House Rent Allowance (20%, partially exempt), Special Allowance (20%, fully taxable), Performance Bonus (10%, fully taxable), Provident Fund (5%, exempt up to limits), Gratuity (3%, exempt up to Rs. 20 lakhs), and Medical Reimbursement (2%, exempt up to Rs. 15,000 p.a.). Additionally, it mentions that the performance appraisal cycle is part of the policy, which supersedes all previous HR policy versions. For queries, employees can contact HR via email or the HR portal.'

In [12]:
print(ans2)

The document outlines the paternity leave policy and the salary structure at NovaTech. The salary structure includes various components: Basic Salary (40% of CTC, fully taxable), House Rent Allowance (20%, partially exempt), Special Allowance (20%, fully taxable), Performance Bonus (10%, fully taxable), Provident Fund (5%, exempt up to limits), Gratuity (3%, exempt up to Rs. 20 lakhs), and Medical Reimbursement (2%, exempt up to Rs. 15,000 p.a.). Additionally, it mentions that the performance appraisal cycle is part of the policy, which supersedes all previous HR policy versions. For queries, employees can contact HR via email or the HR portal.


In [14]:
for i, d in enumerate(srcs2,1):
    print("###")
    print(f"{i}. {d.page_content}")

###
1. to documentation.
2.4 Paternity Leave
###
2. Section 3: Compensation and Benefits
3.1 Salary Structure
The salary structure at NovaTech comprises the following components:
Component
% of CTC
Tax Treatment
Basic Salary
40%
Fully Taxable
House Rent Allowance (HRA)
20%
Partially Exempt (Sec 10(13A))
Special Allowance
20%
Fully Taxable
Performance Bonus
10%
Fully Taxable
Provident Fund (Employer)
5%
Exempt up to limits
Gratuity (Employer)
3%
Exempt up to Rs. 20 lakhs
Medical Reimbursement
2%
Exempt up to Rs. 15,000 p.a.
3.2 Performance Appraisal Cycle
###
3. This policy supersedes all previous HR policy versions. Queries: hr@novatech.in or raise a ticket on the HR
portal.


In [ ]:
files =["data_test/hr_policy_manual.pdf",
        "data_test/novacrm_product_manual.pdf",
        "data_test/annual_financial_report_2024.pdf",
        ""]

all_pages = []
for pdf in files:
    loader=PyPDFLoader(pdf)
    pages = loader.load()
    all_pages.extend(pages)



splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ".", ","],
    chunk_size=512,
    chunk_overlap=100,
)
chunks = splitter.split_documents(all_pages)
vectorstore = FAISS.from_documents(chunks, embeddings)

ret_all    = vectorstore.as_retriever(search_kwargs={"k": 5})

chain_all = (
    {"context": ret_all | fmt, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)


In [17]:
len(all_pages)

13

In [19]:
q = "What was the company's performance last year and what are employee benefits?"
ans4  = chain_all.invoke(q)

In [21]:
print(ans4)

In FY 2023-24, NovaTech reported a total revenue of Rs. 284 Crore, reflecting a year-over-year growth of 28.5%. The gross profit increased to Rs. 142 Crore, marking a growth of 37.9%. EBITDA rose to Rs. 61 Crore, with a growth of 45.2%, and net profit reached Rs. 38 Crore, up by 58.3%. The EBITDA margin improved to 21.5%, and the net profit margin increased to 13.4%. The employee count at the end of the year was 1,842, which is a 32.4% increase from the previous year.

Regarding employee benefits, the salary structure includes several components: Basic Salary (40% of CTC, fully taxable), House Rent Allowance (20%, partially exempt), Special Allowance (20%, fully taxable), Performance Bonus (10%, fully taxable), Provident Fund (5%, exempt up to limits), Gratuity (3%, exempt up to Rs. 20 lakhs), and Medical Reimbursement (2%, exempt up to Rs. 15,000 p.a.). Additionally, the company conducts annual performance appraisals in April, with salary revisions communicated by March 31, and offers

In [23]:
srcs4 = ret_all.invoke(q)
for i, d in enumerate(srcs4,1):
    print("###")
    print(f"{i}. {d.page_content}")
    print(d.metadata)


###
1. Medical Reimbursement
2%
Exempt up to Rs. 15,000 p.a.
3.2 Performance Appraisal Cycle
Appraisals are conducted annually in April. Process: self-assessment, manager rating, calibration.
Salary revisions effective 1st April are communicated by March 31. Mid-year promotions may be
considered in October for exceptional performers.
3.3 Employee Provident Fund (EPF)
NovaTech contributes 12% of Basic Salary towards EPF per the EPF Act, 1952. Employee
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-31T19:53:57+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-31T19:53:57+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'data_test/hr_policy_manual.pdf', 'total_pages': 4, 'page': 2, 'page_label': '3'}
###
2. Section 3: Cost Structure and Profitability
3.1 Operating Expenses Breakdown
Employee costs are the largest expense at 48% of total operating costs (Rs. 117 Crore FY2

In [24]:
q2   = "Give me a summary of all three documents."
ans2 = chain_all.invoke(q2)
print(ans2)

The documents cover various policies and features related to employee management and marketing automation. 

1. **Paternity Leave**: This section outlines the support for pricing structures, multi-currency quotes, product bundling, and volume-based discounts. It mentions the generation of branded PDF proposals and the use of e-signature services like Aadhaar eSign and DocuSign.

2. **Bullying and Discrimination**: Complaints regarding bullying and discrimination must be submitted to the Internal Complaints Committee (ICC) within three months of the incident, with the ICC required to complete inquiries within 90 days.

3. **Confidentiality and Data Security**: Employees are required to sign a Non-Disclosure Agreement (NDA) before joining, and confidential data cannot be shared externally without legal approval. Violations can lead to termination and legal consequences under the IT Act, 2000.

4. **Disciplinary Action**: This section outlines the response times for different levels of di

In [25]:
srcs5 = ret_all.invoke(q2)



In [27]:
for i, d in enumerate(srcs5,1):
    print("###")
    print(f"{i}. {d.page_content}")
    print(d.metadata['source'])


###
1. to documentation.
2.4 Paternity Leave
data_test/hr_policy_manual.pdf
###
2. Supports GST-inclusive and GST-exclusive pricing, multi-currency quotes (INR, USD, AED, SGD),
product bundling, and volume-based discount slabs. Proposals generated as branded PDFs from
configurable templates. E-signature via Aadhaar eSign and DocuSign.
data_test/novacrm_product_manual.pdf
###
3. bullying, and discrimination. Complaints must be submitted to the Internal Complaints Committee
(ICC) within 3 months of the incident. ICC completes inquiry within 90 days.
4.2 Confidentiality and Data Security
All employees must sign the NDA before joining. Confidential data must not be shared externally
without written Legal approval. Violations may result in immediate termination and legal action under
the IT Act, 2000.
4.3 Disciplinary Action
data_test/hr_policy_manual.pdf
###
4. P3 Medium
4 hours
24 hours
Team Lead
P4 Low
24 hours
72 hours
Auto-resolve
3.3 Knowledge Base
Built-in Knowledge Base supports art